# 06g — Inventory Simulation

**Purpose:** Convert forecasts/policies into reorder decisions and simulate business outcomes
on the Fold 2 val window. Section 1 builds the one piece that's still missing — a
`reorder_point`/`order_qty` table for Smooth/Erratic — and merges it with Lumpy/Intermittent
(06f) into a single table covering all SKUs. Section 2 (next) runs `simulate_inventory()`
against it.

**Inputs:**
- `unified_lumpy_intermittent_fold2.parquet` — 06f, Lumpy + Intermittent, already final
- `conformal_residuals_fold2.pkl` — 06e, per-SKU residuals + locked service level(s) for Smooth/Erratic
- `tweedie_optimized_fold2.txt` + `features_train_v2.parquet` — 06d/04b, to regenerate point forecasts (see gap note below)
- `sku_regimes_fold2.parquet` — 06c, routing + regime labels

**Outputs:**
- `final_reorder_params_fold2.parquet` — all ~30,490 SKUs, one schema, ready for Section 2

**Two gaps vs. the plan spec, both handled below:**
1. `new_plan.md`'s Pre-flight #1 assumes `tweedie_optimized_predictions_fold2.parquet` exists
   as an input. It doesn't — 06e's save cell (Section 5) persisted `sku_residuals` and run
   metadata only, never the per-row `yhat` frame it built in memory. Point forecasts are
   regenerated below using 06e Section 1's exact logic, with a row-count sanity check against
   06e's own residual counts to confirm the regenerated population matches what the residuals
   were calibrated against.
2. **Service level is regime-specific, not one locked value.** 06e Section 4 (cell 8) picks a
   *different* service level for Smooth vs. Erratic via an elbow-detection heuristic on
   interval width — `conformal['default_service_level']` is a dict (`{'smooth': 'qXX',
   'erratic': 'qXX'}`), not the single flat 0.80 that Lumpy/Intermittent locked to. The first
   draft of this notebook assumed a scalar and crashed on it. Fixed below by threading the
   per-regime level through the loop, and by adding an explicit `service_level` column to the
   final schema (0.80 for Lumpy/Intermittent, per-regime for Smooth/Erratic) so Section 5 can
   report it honestly instead of implying one level applies everywhere.

---

## Section 1 — Reorder Parameter Computation (Smooth/Erratic) + Three-Way Merge


In [1]:
# ── Setup — load 06e's conformal output, regenerate the point forecasts it never saved ──
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR    = '../data/processed'
CALIBRATION_DIR  = f'{PROCESSED_DIR}/calibration'
SEGMENTATION_DIR = f'{PROCESSED_DIR}/segmentation'
MODELS_DIR       = f'{PROCESSED_DIR}/models'
PREDICTIONS_DIR  = f'{PROCESSED_DIR}/predictions'

LEAD_TIME_DAYS = 7   # same lead time 06f Section 6/7 locked for Lumpy + Intermittent

with open(f'{CALIBRATION_DIR}/conformal_residuals_fold2.pkl', 'rb') as f:
    conformal = pickle.load(f)

sku_residuals = conformal['sku_residuals']
# NOT a single float -- 06e Section 4 picks a service level PER REGIME (elbow-detection on
# interval width), e.g. {'smooth': 'q80', 'erratic': 'q90'}. Converted to a fraction below.
SERVICE_LEVEL_BY_REGIME = conformal['default_service_level']
RESIDUAL_UNIT = conformal['residual_unit']
WINNER_MODEL  = conformal['winner_model']

def service_level_frac(regime):
    """'q80' -> 0.80. Falls back to 0.80 if a regime is missing from the dict."""
    label = SERVICE_LEVEL_BY_REGIME.get(regime, 'q80')
    return int(label[1:]) / 100

print(f'Loaded conformal residuals for {len(sku_residuals):,} SKUs.')
print(f'Residual unit: {RESIDUAL_UNIT}')
print(f'Locked service levels by regime: {SERVICE_LEVEL_BY_REGIME}')

# SANITY CHECK, not a blocker: 06e Section 1 filtered to Tweedie-routed SKUs only (~9,196
# expected). If len(sku_residuals) above is closer to the FULL SKU population (~30,490),
# 06e's scope may have changed since last reviewed -- worth confirming on your end. The loop
# below only ever iterates `tweedie_skus` regardless, so this can't corrupt this notebook's
# output, but it's a signal something upstream may be worth a second look.

# Which 06d experiments trained on a 7-day-forward-sum target vs. next-day -- same check
# 06e Section 1 used, kept here so the branch below can't silently drift from what the
# residuals were actually calibrated against.
SEVEN_DAY_TARGET_EXPERIMENTS = {'experiment_a', 'experiment_c'}
TARGET_IS_7DAY = WINNER_MODEL in SEVEN_DAY_TARGET_EXPERIMENTS
assert (RESIDUAL_UNIT == 'expected demand, rolling next-7-days') == TARGET_IS_7DAY, (
    'Winner model / residual unit mismatch vs. 06e -- regenerating yhat below would use the '
    'wrong native-space branch. Stop and check 06e Section 1 before proceeding.'
)

FOLD2_VAL_START = '2014-02-01'
CALIBRATION_END = '2014-08-31'   # same calibration window the residuals were built on

model = lgb.Booster(model_file=f'{MODELS_DIR}/tweedie_optimized_fold2.txt')
sku_regimes = pd.read_parquet(f'{SEGMENTATION_DIR}/sku_regimes_fold2.parquet')
tweedie_skus = set(sku_regimes.loc[sku_regimes['routing'] == 'tweedie', 'id'])
regime_map = sku_regimes.set_index('id')['regime'].to_dict()

print(f'Tweedie-routed SKUs (Smooth+Erratic, from 06c routing): {len(tweedie_skus):,}')

train_full = pd.read_parquet(f'{PROCESSED_DIR}/features/features_train_v2.parquet')
train_full['date'] = pd.to_datetime(train_full['date'])
with open(f'{PROCESSED_DIR}/features/feature_cols_v2.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

mask_se    = train_full['id'].isin(tweedie_skus)
mask_calib = (train_full['date'] >= FOLD2_VAL_START) & (train_full['date'] <= CALIBRATION_END)
calib_se = train_full[mask_se & mask_calib].sort_values(['id', 'date']).copy()
calib_se['yhat'] = np.maximum(model.predict(calib_se[feature_cols].values), 0)

print(f"Regenerated yhat for {calib_se['id'].nunique():,} Smooth/Erratic SKUs "
      f'over the calibration window ({len(calib_se):,} rows).')

# Sanity check: regenerated population should match 06e's residual counts row-for-row for
# the Tweedie-routed SKUs specifically -- same window, same SKUs, same model. A mismatch
# here means these yhat values aren't the same population the residuals were calibrated
# against, and nothing below can be trusted.
residual_counts = pd.Series({k: len(v) for k, v in sku_residuals.items() if k in tweedie_skus})
row_counts = calib_se.groupby('id', observed=True).size()
common = residual_counts.index.intersection(row_counts.index)
mismatch = (residual_counts.loc[common] != row_counts.loc[common]).sum()
print(f'Tweedie SKUs with mismatched row/residual counts vs. 06e: {mismatch:,} / {len(common):,}')

missing_from_calib = sorted(tweedie_skus - set(row_counts.index))
print(f'Tweedie-routed SKUs with zero rows in the calibration window: {len(missing_from_calib):,}')
if missing_from_calib:
    print('  These have no history to forecast from -- caught by the NaN-forecast guard in the')
    print('  next cell (empty-group mean -> NaN), not excluded via a point_forecast.index check.')

# BUG FOUND + FIXED: 'id' is a categorical dtype, so groupby('id') without observed=True keeps
# every one of the full 30,490 SKU categories in its output index -- including the ~21,271 that
# have zero rows in calib_se after filtering. That silently broke the two checks above (both
# were structurally guaranteed to read '0'/'all-present' regardless of the real data) and meant
# the 'excluded here' comment on missing_from_calib was never actually true -- those SKUs fell
# through to point_forecast in the next cell instead. observed=True above fixes row_counts here;
# the same fix is applied to point_forecast in the next cell.


Loaded conformal residuals for 30,490 SKUs.
Residual unit: expected demand, next-day
Locked service levels by regime: {'smooth': 'q80', 'erratic': 'q80'}
Tweedie-routed SKUs (Smooth+Erratic, from 06c routing): 9,219
Regenerated yhat for 9,171 Smooth/Erratic SKUs over the calibration window (1,806,148 rows).
Tweedie SKUs with mismatched row/residual counts vs. 06e: 0 / 9,171
Tweedie-routed SKUs with zero rows in the calibration window: 48
  These have no history to forecast from -- caught by the NaN-forecast guard in the
  next cell (empty-group mean -> NaN), not excluded via a point_forecast.index check.


In [2]:
# ── compute_reorder_params — Smooth/Erratic only (Lumpy/Intermittent read directly in the ──
# merge cell below, per Pre-flight #1; nothing here touches those two regimes)
MIN_RESIDUALS_PER_SKU = 5   # same floor 06e Section 2 used for a stable per-SKU quantile

point_forecast = calib_se.groupby('id', observed=True)['yhat'].mean()   # "typical demand," same framing as
                                                          # Lumpy's avg_weekly / Intermittent's
                                                          # mean_lead_time_demand -- a single
                                                          # static value per SKU, not the most
                                                          # recent day's forecast (Pre-flight #3
                                                          # needs a fixed value, not a moving one)

# Pooled fallback for SKUs with too few calibration residuals for their own stable quantile.
# Built PER REGIME now, not globally pooled -- Smooth and Erratic use different service
# levels, so a single global pooled quantile would apply the wrong level to half the fallback.
pooled_residuals_by_regime = {}
for regime in ('smooth', 'erratic'):
    ids_in_regime = {sid for sid, r in regime_map.items() if r == regime}
    pooled = np.concatenate([
        np.asarray(v) for sid, v in sku_residuals.items() if sid in ids_in_regime
    ]) if ids_in_regime else np.array([0.0])
    pooled_residuals_by_regime[regime] = pooled

rows = []
n_negative_q80 = 0
n_nan_forecast = 0
n_zero_history_excluded = 0
nan_forecast_ids = []
for sku_id in sorted(tweedie_skus):
    if sku_id not in point_forecast.index:
        # With observed=True fixed above, this now actually fires for the zero-history
        # cohort instead of being a dead branch -- explicitly excluded here, not routed
        # through the NaN-forecast guard below.
        n_zero_history_excluded += 1
        continue

    regime = regime_map[sku_id]
    sl = service_level_frac(regime)   # regime-specific, not a flat 0.80

    pf_native = point_forecast[sku_id]
    residuals = sku_residuals.get(sku_id, [])
    low_conf  = len(residuals) < MIN_RESIDUALS_PER_SKU
    if low_conf:
        resid_q = np.percentile(pooled_residuals_by_regime.get(regime, [0.0]), sl * 100)
    else:
        resid_q = np.percentile(residuals, sl * 100)
    if resid_q < 0:
        n_negative_q80 += 1
    resid_q = max(resid_q, 0.0)   # a SKU that mostly over-forecasts gets no negative "safety
                                   # debt" -- floored the same way Intermittent's reactive-only
                                   # cohort was in 06f, not left as-is

    # NOTE: residual = target - yhat already, so `resid_q` IS the interval half-width
    # (equivalent to the plan's `q_upper - point_forecast`) -- no second subtraction needed.
    if TARGET_IS_7DAY:
        # Model already predicts a native 7-day-forward sum -- no daily->lead-time scaling.
        expected_lead_time_demand = pf_native
        safety_stock = resid_q
    else:
        # Model predicts next-day units -- scaling to the 7-day lead time via sqrt(7).
        # UNVALIDATED (Pre-flight #4): assumes i.i.d. daily forecast errors. 06f Section 6
        # found the opposite for Intermittent (real demand clusters in time, ~2.9x more
        # lead-time variance than an independent-day model implies). Smooth/Erratic demand is
        # more regular and may hold up better, but that hasn't been checked with the same
        # rolling-origin coverage test yet -- do not treat these reorder points as trustworthy
        # until that runs.
        expected_lead_time_demand = pf_native * LEAD_TIME_DAYS
        safety_stock = resid_q * np.sqrt(LEAD_TIME_DAYS)

    # GUARD: Python's built-in max() does NOT floor NaN -- max(nan, 1.0) returns nan, because
    # every comparison against nan is False. If the point forecast came out NaN/inf (e.g. a
    # SKU with too little lag-feature history in the calibration window), silently taking
    # max(nan, 1.0) would let a NaN slip into order_qty and fail the assert below with no
    # visibility into which SKU or why. Caught explicitly instead: treated as low-confidence/
    # fallback (same as the zero-history cohort), not guessed at.
    if not (np.isfinite(expected_lead_time_demand) and np.isfinite(safety_stock)):
        n_nan_forecast += 1
        nan_forecast_ids.append(sku_id)
        low_conf = True
        fallback_required = True
        expected_lead_time_demand = 0.0
        safety_stock = 0.0
        reorder_point = 0.0
        order_qty = 1.0
    else:
        fallback_required = low_conf
        reorder_point = expected_lead_time_demand + safety_stock
        order_qty = max(expected_lead_time_demand, 1.0)   # never order zero -- same floor logic as 06f

    rows.append({
        'id':                        sku_id,
        'regime':                    regime,
        'method':                    'conformal_tweedie',
        'expected_lead_time_demand': expected_lead_time_demand,
        'reorder_point':             reorder_point,
        'safety_buffer':             safety_stock,
        'order_qty':                 order_qty,
        'low_confidence':            low_conf,
        'fallback_required':         fallback_required,
        'sqrt_scaling_unvalidated':  not TARGET_IS_7DAY,
        'service_level':             sl,
    })

smooth_erratic_unified = pd.DataFrame(rows)

print(f'Zero-history SKUs excluded (no rows in calibration window): {n_zero_history_excluded:,}')
print(f'SKUs with NaN/inf point forecast (guarded, forced to fallback): {n_nan_forecast:,}')
if nan_forecast_ids:
    print(f'  Sample IDs: {nan_forecast_ids[:10]}')
    print('  Worth checking whether these share a cause (e.g. launched too recently for full')
    print('  lag-feature history in the calibration window) before trusting the fallback value.')

assert smooth_erratic_unified['order_qty'].notna().all() and (smooth_erratic_unified['order_qty'] > 0).all(), (
    'BUG: some Smooth/Erratic SKU has order_qty <= 0 -- would never reorder in simulation.'
)

print(f'Smooth/Erratic reorder table: {len(smooth_erratic_unified):,} SKUs.')
print(f'Low-confidence (<{MIN_RESIDUALS_PER_SKU} calib residuals OR NaN forecast): '
      f"{smooth_erratic_unified['low_confidence'].sum():,}")
print(f'SKUs where the raw q_level residual was negative before flooring: {n_negative_q80:,} '
      f'(over-forecasting cohort -- reactive-only, same shape as Intermittent\'s cohort in 06f)')
print()
print(smooth_erratic_unified.groupby('regime')[
    ['expected_lead_time_demand', 'reorder_point', 'safety_buffer', 'order_qty', 'service_level']
].describe().round(2))


Zero-history SKUs excluded (no rows in calibration window): 48
SKUs with NaN/inf point forecast (guarded, forced to fallback): 0
Smooth/Erratic reorder table: 9,171 SKUs.
Low-confidence (<5 calib residuals OR NaN forecast): 0
SKUs where the raw q_level residual was negative before flooring: 655 (over-forecasting cohort -- reactive-only, same shape as Intermittent's cohort in 06f)

        expected_lead_time_demand                                         \
                            count   mean    std   min   25%   50%    75%   
regime                                                                     
erratic                     827.0  17.32  29.00  0.94  4.04  7.60  17.03   
smooth                     8344.0  14.89  29.37  0.68  3.77  6.79  14.21   

                reorder_point         ... order_qty         service_level  \
            max         count   mean  ...       75%     max         count   
regime                                ...                                   
erra

In [3]:
# ── Cadence check + three-way merge ──────────────────────────────────────────────────────
# Mirrors 06f Section 7's cadence assert before it concatenated Lumpy + Intermittent --
# confirms Smooth/Erratic's regenerated 7-day lead-time figures and Lumpy/Intermittent's
# weekly figures are actually on the same clock before treating them as comparable.
assert LEAD_TIME_DAYS == 7, (
    'Smooth/Erratic reorder points above assume a 7-day lead time. Lumpy (06f Section 4, '
    'lead_time_weeks=1) and Intermittent (06f Section 6, LEAD_TIME_DAYS=7) both assumed the '
    'same -- if LEAD_TIME_DAYS changes here, the three regimes are no longer on the same '
    'clock and cannot be merged as-is.'
)

lumpy_intermittent = pd.read_parquet(f'{PREDICTIONS_DIR}/unified_lumpy_intermittent_fold2.parquet')
lumpy_intermittent['sqrt_scaling_unvalidated'] = False   # policy / block-bootstrap methods --
                                                          # no i.i.d.-daily-error assumption involved
lumpy_intermittent['service_level'] = 0.80               # locked flat value, per 06f Section 6

final_reorder_params = pd.concat([lumpy_intermittent, smooth_erratic_unified], ignore_index=True)

expected_total = sku_regimes['id'].nunique()
actual_total   = len(final_reorder_params)
print(f'Merged table: {actual_total:,} SKUs vs. {expected_total:,} in 06c\'s full routing '
      f'({expected_total - actual_total:,} short -- should equal the zero-calibration-history '
      f'count printed above, if any).')

assert final_reorder_params['order_qty'].notna().all() and (final_reorder_params['order_qty'] > 0).all(), (
    'order_qty must be populated and positive for every SKU across all three regimes before '
    'Section 2 can run simulate_inventory().'
)

print()
print(final_reorder_params['regime'].value_counts())
print()
print('Service level in effect, by regime:')
print(final_reorder_params.groupby('regime')['service_level'].agg(['min', 'max', 'mean']).round(2))
print()
print(f'Low-confidence / fallback SKUs by regime (Smooth/Erratic only -- Lumpy/Intermittent '
      f'have their own flags from 06f):')
print(final_reorder_params.groupby('regime')[['low_confidence', 'fallback_required']].mean().round(3) * 100)

os.makedirs(PREDICTIONS_DIR, exist_ok=True)
final_reorder_params.to_parquet(f'{PREDICTIONS_DIR}/final_reorder_params_fold2.parquet')
print(f'\n✓ final_reorder_params_fold2.parquet saved ({actual_total:,} SKUs, all regimes).')
print('REMINDER: Smooth/Erratic reorder points carry an unvalidated sqrt(lead_time) daily-error')
print('assumption (Pre-flight #4, sqrt_scaling_unvalidated=True) -- run the rolling-origin')
print('coverage check before Section 2 trusts them in the simulation.')


Merged table: 30,442 SKUs vs. 30,490 in 06c's full routing (48 short -- should equal the zero-calibration-history count printed above, if any).

regime
intermittent    14268
smooth           8344
lumpy            7003
erratic           827
Name: count, dtype: int64

Service level in effect, by regime:
              min  max  mean
regime                      
erratic       0.8  0.8   0.8
intermittent  0.8  0.8   0.8
lumpy         0.8  0.8   0.8
smooth        0.8  0.8   0.8

Low-confidence / fallback SKUs by regime (Smooth/Erratic only -- Lumpy/Intermittent have their own flags from 06f):
              low_confidence  fallback_required
regime                                         
erratic                  0.0                0.0
intermittent             0.0                0.0
lumpy                   59.8               59.8
smooth                   0.0                0.0

✓ final_reorder_params_fold2.parquet saved (30,442 SKUs, all regimes).
REMINDER: Smooth/Erratic reorder points carry 

#### Section 1 Findings — Reorder Parameter Computation + Three-Way Merge

**Bug fixed:** `id` is a categorical column; `groupby('id')` without `observed=True` was silently spanning all 30,490 SKU categories instead of the filtered subset, making two sanity checks structurally unable to fail. Fixed with `observed=True`; diagnostic cell confirms it (`row_counts entries` now equals `nunique`, both 9,171).

**Population:** 9,219 Tweedie-routed SKUs (8,389 smooth + 830 erratic) → **48 excluded** (zero calibration-window history) → **9,171 in the final reorder table** (8,344 smooth + 827 erratic). `n_nan_forecast` dropped to **0**, confirming the 48 "NaN forecast" SKUs from the buggy run were this same zero-history cohort, not a separate issue.

**Merge:** 30,442 SKUs total (14,268 intermittent + 8,344 smooth + 7,003 lumpy + 827 erratic) = 30,490 − 48, exactly as predicted.

**Low-confidence/fallback:** 0% for Smooth/Erratic now (the 48 that used to inflate this count are cleanly excluded instead). Lumpy's 59.8% is carried over from 06f, unrelated to this fix.

**Unaffected by the bug, stands as before:** 655 SKUs with negative raw residual quantile (reactive-only cohort), service levels flat at 0.80 across all regimes, `sqrt(lead_time)` scaling for Smooth/Erratic **still unvalidated** (Pre-flight #4 — not resolved here, still a caveat on Section 2's Smooth/Erratic results specifically).

**Output:** `final_reorder_params_fold2.parquet`, 30,442 SKUs, saved.

## Section 2 — Inventory Depletion Simulation

**Purpose:** Simulate week-by-week inventory under the locked reorder policy from Section 1
(`final_reorder_params_fold2.parquet`), against **actual** sales in the Fold 2 val window —
`simulate_inventory()` per `new_plan.md`'s Phase E spec.

**Decisions made explicit here, not defaulted silently (mirrors the Pre-flight discipline from
Section 1):**

1. **Val window:** `2014-02-01` -> `2015-01-31`, the full Fold 2 val period per the plan's
   Section 2 spec ("Feb 2014 -> Jan 2015"). Different from Section 1's `CALIBRATION_END`
   (`2014-08-31`), which was only the conformal calibration sub-window -- this is the full
   held-out year the simulation actually evaluates against.
2. **Reorder quantity is static**, sourced directly from `order_qty` in the unified table for
   all three regimes -- this is Pre-flight #3's resolution (fixed-quantity policy, not
   order-up-to-S), already locked in Section 1's schema.
3. **`initial_inventory` was never specified in the plan** -- resolved here as
   `reorder_point + order_qty`, i.e. simulate starting from a freshly-stocked shelf (holding the
   reorder point's buffer plus one full incoming replenishment already on the shelf). This is a
   real assumption with real consequences for early-simulation stockout rates and needs to be
   named, the same way order_qty's definition was in Section 1 -- not left as an implicit
   default inside the loop.
4. **Lead time:** `LEAD_TIME_WEEKS = 1`, consistent with the `LEAD_TIME_DAYS == 7` assert
   already locked across all three regimes in Section 1's cadence check.
5. **This section runs the simulation once, at each SKU's already-locked service level**
   (q80 flat for Lumpy/Intermittent, per-regime q80 for Smooth/Erratic) -- not yet the
   multi-service-level sweep (`new_plan.md` Section 3) or the cost tradeoff curve (Section 4).
   Those come after this section's output is confirmed correct.

In [7]:
# ── 06g Section 2: Inventory Depletion Simulation ────────────────────────────────────────
RAW_DIR = '../data/raw'
SIM_VAL_START = '2014-02-01'
SIM_VAL_END   = '2015-01-31'   # Fold 2 val window, full year -- see Section 2 intro, decision 1
LEAD_TIME_WEEKS = 1            # matches LEAD_TIME_DAYS == 7 locked in Section 1's cadence check

calendar_df = pd.read_csv(f'{RAW_DIR}/calendar.csv')
calendar_df['date'] = pd.to_datetime(calendar_df['date'])
day_to_date = calendar_df.set_index('d')['date'].to_dict()

raw_sales = pd.read_csv(f'{RAW_DIR}/sales_train_validation.csv')
day_cols_all = [c for c in raw_sales.columns if c.startswith('d_')]

val_day_cols = [
    d for d in day_cols_all
    if d in day_to_date and SIM_VAL_START <= str(day_to_date[d].date()) <= SIM_VAL_END
]
val_dates = pd.to_datetime([day_to_date[d] for d in val_day_cols])
print(f'Fold 2 val window for simulation: {val_dates.min().date()} -> {val_dates.max().date()} '
      f'({len(val_day_cols)} days)')

final_reorder_params = pd.read_parquet(f'{PREDICTIONS_DIR}/final_reorder_params_fold2.parquet')
sim_ids = set(final_reorder_params['id'])
print(f'SKUs entering simulation from Section 1: {len(sim_ids):,}')

# ── Build weekly actual-demand matrix (id x week) directly from raw sales, same source 06f
# used for its own raw zero-fraction sanity check -- avoids reloading the 440MB feature parquet
# for data this simple. ────────────────────────────────────────────────────────────────────
sales_val = raw_sales[raw_sales['id'].isin(sim_ids)].set_index('id')[val_day_cols]
missing_sales_ids = sorted(sim_ids - set(sales_val.index))
print(f'SKUs with no matching row in raw sales file: {len(missing_sales_ids):,}')
if missing_sales_ids:
    print(f'  Sample: {missing_sales_ids[:5]} -- excluded from simulation, see Findings.')

sales_val_t = sales_val.T
sales_val_t.index = val_dates
weekly_actuals = sales_val_t.resample('W').sum().T   # id (rows) x calendar week (cols)
print(f'Weekly actual-demand matrix: {weekly_actuals.shape[0]:,} SKUs x {weekly_actuals.shape[1]} weeks')


def simulate_inventory(actual_weekly_demand: np.ndarray,
                        reorder_point: float,
                        reorder_qty: float,
                        initial_inventory: float,
                        lead_time_weeks: int) -> dict:
    """
    Simulate inventory trajectory under a fixed-quantity reorder policy.
    Returns stockout weeks, average inventory, service level achieved.
    (Verbatim from new_plan.md Phase E Section 2 -- no logic changes.)
    """
    inventory = initial_inventory
    stockout_weeks = 0
    inventory_history = []
    pending_order = 0
    weeks_until_arrival = 0

    for week, demand in enumerate(actual_weekly_demand):
        if weeks_until_arrival == 0 and pending_order > 0:
            inventory += pending_order
            pending_order = 0

        fulfilled = min(inventory, demand)
        if demand > inventory:
            stockout_weeks += 1
        inventory -= fulfilled

        if inventory <= reorder_point and pending_order == 0:
            pending_order = reorder_qty
            weeks_until_arrival = lead_time_weeks

        inventory_history.append(inventory)
        if weeks_until_arrival > 0:
            weeks_until_arrival -= 1

    return {
        'stockout_rate': stockout_weeks / len(actual_weekly_demand),
        'avg_inventory': np.mean(inventory_history),
        'fill_rate': 1 - stockout_weeks / len(actual_weekly_demand),
    }


sim_rows = []
skipped_no_sales = 0
for row in final_reorder_params.itertuples(index=False):
    sku_id = row.id
    if sku_id not in weekly_actuals.index:
        skipped_no_sales += 1
        continue

    demand = weekly_actuals.loc[sku_id].values
    # Decision 3 (Section 2 intro): freshly-stocked-shelf starting assumption, made explicit
    # rather than an implicit zero or an implicit reorder_point-only default.
    initial_inventory = row.reorder_point + row.order_qty

    result = simulate_inventory(
        actual_weekly_demand=demand,
        reorder_point=row.reorder_point,
        reorder_qty=row.order_qty,
        initial_inventory=initial_inventory,
        lead_time_weeks=LEAD_TIME_WEEKS,
    )
    sim_rows.append({
        'id': sku_id,
        'regime': row.regime,
        'service_level': row.service_level,
        'stockout_rate': result['stockout_rate'],
        'avg_inventory': result['avg_inventory'],
        'fill_rate': result['fill_rate'],
    })

assert skipped_no_sales == len(missing_sales_ids), (
    'Loop-level skip count disagrees with the missing_sales_ids check above -- investigate '
    'before trusting simulation_results.'
)

simulation_results = pd.DataFrame(sim_rows)
print()
print(f'Simulated {len(simulation_results):,} SKUs '
      f'({skipped_no_sales:,} skipped -- no raw sales row, see Findings).')
print()
print('── Simulation results by regime, at each regime\'s locked service level ──')
print(simulation_results.groupby('regime')[['stockout_rate', 'avg_inventory', 'fill_rate']]
      .describe().round(3))

os.makedirs(PREDICTIONS_DIR, exist_ok=True)
simulation_results.to_parquet(f'{PREDICTIONS_DIR}/simulation_results_fold2.parquet')
print(f'\n✓ simulation_results_fold2.parquet saved ({len(simulation_results):,} SKUs).')

Fold 2 val window for simulation: 2014-02-01 -> 2015-01-31 (365 days)
SKUs entering simulation from Section 1: 30,442
SKUs with no matching row in raw sales file: 0
Weekly actual-demand matrix: 30,442 SKUs x 53 weeks

Simulated 30,442 SKUs (0 skipped -- no raw sales row, see Findings).

── Simulation results by regime, at each regime's locked service level ──
             stockout_rate                                                 \
                     count   mean    std  min    25%    50%    75%    max   
regime                                                                      
erratic              827.0  0.219  0.118  0.0  0.132  0.208  0.283  0.660   
intermittent       14268.0  0.479  0.324  0.0  0.170  0.453  0.792  1.000   
lumpy               7003.0  0.168  0.244  0.0  0.000  0.038  0.245  1.000   
smooth              8344.0  0.219  0.126  0.0  0.132  0.189  0.283  0.698   

             avg_inventory          ...                   fill_rate         \
                    

### Section 2 Findings — Inventory Depletion Simulation

**[RERUN NEEDED]** -- run Section 1's fix first (cells 1-5), confirm its findings, *then* run
this section fresh, since `final_reorder_params_fold2.parquet` will be re-saved with the
corrected SKU population (~9,171 Smooth/Erratic rows, not 9,219) before this section reads it
back in.

Once rerun, paste:
- SKUs entering simulation vs. SKUs actually simulated (missing-sales count, if any -- should be
  0 or near-0, since `final_reorder_params_fold2.parquet` was itself built from SKUs known to
  exist in the training data)
- The per-regime `stockout_rate` / `avg_inventory` / `fill_rate` describe() table
- Anything that looks off against the sanity expectation from `new_plan.md`: Smooth should show
  high fill rates at its locked service level (model is accurate); Lumpy should show the
  roughest fill rates given its 59.8% low-confidence rate carried over from 06f

**Known limitation carried forward, not resolved by this section:** Smooth/Erratic's
`sqrt(lead_time)` safety-stock scaling is still unvalidated (Pre-flight #4). If Smooth/Erratic's
`stockout_rate` comes in noticeably worse than Lumpy/Intermittent's block-bootstrap-validated
numbers, that's the first place to look -- it would be consistent with (not proof of) the
scaling assumption underestimating lead-time variance, the same failure mode 06f Section 6 found
and fixed for Intermittent before trusting its numbers.